<a href="https://colab.research.google.com/github/vivi0424/HSE-Computational-linguistics/blob/main/deborina_w2v_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [1]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 13.3 MB/s eta 0:00:00


## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [2]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

**Базовые операции с векторами**

In [3]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [4]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


*Ваш ответ здесь*

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [5]:
w2v_model = api.load('word2vec-google-news-300')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

[==================================================] 100.0% 1662.8/1662.8MB downloaded
Размер словаря: 3000000
Размерность векторов: 300


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [6]:
# Получаем вектор слова
vector = w2v_model['cat']
print(f"Вектор слова 'cat': {vector[:5]}...")  # Показываем первые 5 чисел

# Находим похожие слова
similar_words = w2v_model.most_similar('cat', topn=10)
print("Слова, похожие на 'cat':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Вектор слова 'cat': [ 0.0123291   0.20410156 -0.28515625  0.21679688  0.11816406]...
Слова, похожие на 'cat':
  cats: 0.8099
  dog: 0.7609
  kitten: 0.7465
  feline: 0.7326
  beagle: 0.7151
  puppy: 0.7075
  pup: 0.6934
  pet: 0.6892
  felines: 0.6756
  chihuahua: 0.6710


In [8]:
# то же самое через input()
word = input("Введите слово: ").strip()

# проверяем, есть ли слово в словаре модели
if word in w2v_model.key_to_index:
    vector = w2v_model[word]
    print(f"Вектор слова '{word}': {vector[:5]}...")  # показываем первые 5 чисел

    # Находим похожие слова
    similar_words = w2v_model.most_similar(word, topn=10)
    print(f"Слова, похожие на '{word}':")
    for w, score in similar_words:
        print(f"  {w}: {score:.4f}")
else:
    print(f"Слова '{word}' нет в словаре модели :(")

Введите слово: dog
Вектор слова 'dog': [ 0.05126953 -0.02233887 -0.17285156  0.16113281 -0.08447266]...
Слова, похожие на 'dog':
  dogs: 0.8680
  puppy: 0.8106
  pit_bull: 0.7804
  pooch: 0.7627
  cat: 0.7609
  golden_retriever: 0.7501
  German_shepherd: 0.7465
  Rottweiler: 0.7438
  beagle: 0.7419
  pup: 0.7407


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [9]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [11]:
model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,  # размер вектора
    window=3,        # размер окна
    min_count=1,     # мин. частота слова
    workers=2,       # потоков
    sg=1,            # skip-gram
    seed=42
)

In [12]:
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [13]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  горшок: 0.2121
  резать: 0.2099
  салат: 0.2008
  соус: 0.1908
  картофель: 0.1876


In [14]:
# Найдите слова, похожие на "духовка"
similar_words = model.wv.most_similar('духовка', topn=10)
print("Слова, похожие на 'духовка':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

# Найдите слова, похожие на "овощи"
similar_words = model.wv.most_similar('овощи', topn=10)
print("Слова, похожие на 'овощи':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'духовка':
  барбекю: 0.2798
  мариновать: 0.2657
  молоко: 0.2295
  специи: 0.2247
  жарить: 0.1861
  огурцы: 0.1794
  мука: 0.1609
  готовить: 0.1594
  вино: 0.1465
  суп: 0.1410
Слова, похожие на 'овощи':
  вино: 0.3312
  сковорода: 0.2633
  взбивать: 0.2424
  ингредиенты: 0.2323
  рыба: 0.2130
  бекон: 0.2044
  травы: 0.1548
  хлеб: 0.1486
  тост: 0.1386
  здоровое: 0.1313


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [16]:
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [20]:
# Найдите слова, похожие на "варить"
similar_words = ft_model.wv.most_similar('варить', topn=10)
print("Слова, похожие на 'варить':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

# Найдите слова, похожие на "духовка"
similar_words = ft_model.wv.most_similar('духовка', topn=10)
print("Слова, похожие на 'духовка':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

# Найдите слова, похожие на "овощи"
similar_words = ft_model.wv.most_similar('овощи', topn=10)
print("Слова, похожие на 'овощи':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'варить':
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622
  рыба: 0.2425
  бекон: 0.2348
  запекать: 0.2155
  завтрак: 0.2032
  смешивать: 0.1994
Слова, похожие на 'духовка':
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944
  кофе: 0.2896
  месить: 0.2676
  говядина: 0.2456
  масло: 0.2428
  рыба: 0.2423
Слова, похожие на 'овощи':
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094
  здоровое: 0.1955
  ингредиенты: 0.1942
  кипятить: 0.1651
  сахар: 0.1448
  барбекю: 0.1434


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [25]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
compare_models('рокмовь')
compare_models('шуп')
compare_models('хакар')


Сравнение для слова: 'рокмовь'
  Word2Vec: слово не найдено
  FastText: ['брокколи', 'уголь']

Сравнение для слова: 'шуп'
  Word2Vec: слово не найдено
  FastText: ['взбивать', 'пирог']

Сравнение для слова: 'хакар'
  Word2Vec: слово не найдено
  FastText: ['соль', 'пирог']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [26]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [27]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [28]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [29]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [34]:
def compare_documents(doc2_id, doc4_id):
    similarity = doc_model.dv.similarity(f"doc_{doc2_id}", f"doc_{doc4_id}")
    print(f"Схожесть doc_{doc2_id} и doc_{doc4_id}: {similarity:.4f}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")
    print(f"  doc_{doc4_id}: {documents[doc4_id]}")

compare_documents(2, 4)

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [35]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_1"]
print(f"Вектор документа doc_1: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_1", topn=1)
print("\nДокументы, похожие на doc_1:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_1: [-0.00435809 -0.00022557 -0.01330682 -0.01307992 -0.00394024]...

Документы, похожие на doc_1:
  doc_0: 0.2735
    Текст: machine learning is interesting


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [38]:
ft_model_10 = FastText(
    sentences=cooking_sentences,
    vector_size=10,
    window=3,
    min_count=1,
    workers=2
)

In [39]:
ft_model_50 = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

In [40]:
ft_model_100 = FastText(
    sentences=cooking_sentences,
    vector_size=100,
    window=3,
    min_count=1,
    workers=2
)

In [41]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # FastText (10)
    try:
        ft1_similar = ft_model_10.wv.most_similar(word, topn=2)
        print(f"  FastText (10): {[w for w, _ in ft1_similar]}")
    except KeyError:
        print(f"  FastText (10): слово не найдено")

    # FastText (50)
    try:
        ft2_similar = ft_model_50.wv.most_similar(word, topn=2)
        print(f"  FastText (50): {[w for w, _ in ft2_similar]}")
    except KeyError:
        print(f"  FastText (50): слово не найдено")

    # FastText (100)
    try:
        ft3_similar = ft_model_100.wv.most_similar(word, topn=2)
        print(f"  FastText (100): {[w for w, _ in ft3_similar]}")
    except KeyError:
        print(f"  FastText (100): слово не найдено")


# Сравниваем для разных слов
compare_models('варенье')
compare_models('хлеб')
compare_models('вишня')


Сравнение для слова: 'варенье'
  FastText (10): ['печь', 'мариновать']
  FastText (50): ['вино', 'помидоры']
  FastText (100): ['морковь', 'варить']

Сравнение для слова: 'хлеб'
  FastText (10): ['завтрак', 'бекон']
  FastText (50): ['молоко', 'паста']
  FastText (100): ['бекон', 'картофель']

Сравнение для слова: 'вишня'
  FastText (10): ['завтрак', 'кипятить']
  FastText (50): ['торт', 'травы']
  FastText (100): ['мука', 'дрожжи']
